
既然你的硬盘已经挂载在 /mnt/ex_8T_SSD 且实测写入速度达到 420MB/s，在这个盘上设置 Swap 是非常安全的。
请按以下 5 个步骤 依次执行。
第一步：创建 Swap 文件
我们要创建一个 64GB 的大文件。使用 fallocate 命令最快（几乎瞬间完成）：

In [ ]:
sudo fallocate -l 64G /mnt/ex_8T_SSD/swapfile

第二步：设置安全权限
Linux 要求交换文件必须只能由 root 用户读写，否则不会启动。

In [ ]:
sudo chmod 600 /mnt/ex_8T_SSD/swapfile

In [ ]:
第三步：格式化为交换分区格式
告知系统这个文件是用来做 Swap 的。

In [ ]:
sudo mkswap /mnt/ex_8T_SSD/swapfile

第四步：激活 Swap
正式开启交换功能。

In [ ]:
sudo swapon /mnt/ex_8T_SSD/swapfile

第五步：验证是否成功
检查系统现在的总内存和 Swap 情况：

In [ ]:
swapon --show

你应该能看到类似这样的输出：
code
Text
NAME                     TYPE  SIZE   USED  PRIO
/mnt/ex_8T_SSD/swapfile  file  64G    0B    -2
或者运行 free -h，看 Swap 那一行的 Total 是否增加了约 64G。

⚠️ 针对生信分析的两个重要优化：
1. 调整 Swappiness（重要！）
默认情况下，Linux 会很“积极”地使用 Swap。但我们希望只有在 125G 内存真的不够用时才用 Swap，因为物理内存比 SSD 快得多。
运行以下命令，将内核的 Swap 积极性从 60 降到 10：
code
Bash
sudo sysctl vm.swappiness=10
(这在重启前一直有效。如果你想永久生效，需在 /etc/sysctl.conf 末尾添加 vm.swappiness=10)

2. 关于重启后的处理（特别提醒）
由于你的硬盘是外置的，不要把这个 Swap 写进 /etc/fstab（自动挂载文件）。
原因： 如果哪天你开机时没插这个移动硬盘，或者挂载慢了，系统由于找不到 fstab 里的 Swap 文件，可能会卡在开机界面无法进入系统。
建议： 每次你需要跑 12 个 WGS 任务前，手动运行一下 sudo swapon /mnt/ex_8T_SSD/swapfile 即可。